<a href="https://colab.research.google.com/github/d-montenegro/eph-agente/blob/main/notebooks/eph_1t2026_sqlite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EPH 1T 2026: descarga, filtros y SQLite

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/d-montenegro/eph-agente/blob/main/notebooks/eph_1t2026_sqlite.ipynb)

Este cuaderno es la primera prueba del flujo de datos: baja los microdatos del INDEC, permite trabajar con una parte si hace falta y arma una base SQLite. El ZIP original no se modifica.

Por ahora trabajamos sólo con la EPH continua del **primer trimestre de 2026**. Fuente: Instituto Nacional de Estadística y Censos (INDEC).

## 1. Configuración

Para arrancar, podés dejar `AGLOMERADOS` vacío y las columnas en `None`: así se carga la base completa. Si sólo querés hacer una prueba rápida, elegí uno o más aglomerados o una lista de columnas. Las columnas necesarias para identificar cada registro se agregan solas.

Ojo: si aplicás filtros, la base resultante ya no representa a todo el país. Sirve para probar el proceso, pero cualquier cálculo tiene que aclarar qué filtro se usó.

In [11]:
from __future__ import annotations

import hashlib
import io
import json
import os
import sqlite3
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

URLS_FUENTE = [
    "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/EPH_usu_1_Trim_2026_txt.zip",
    # Si la web del INDEC no responde, probamos con la copia guardada en el repo.
    "https://raw.githubusercontent.com/d-montenegro/eph-agente/main/datos_indec/EPH_usu_1_Trim_2026_txt.zip",
]
EXPECTED_ZIP_SHA256 = "d9d23945d4b1d3de362e154015fd9abacc76ed641aec6f41a805b809780c0343"
ZIP_PATH = Path("/content/EPH_usu_1_Trim_2026_txt.zip")
DB_PATH = Path("/content/eph_1t2026.sqlite")
MANIFEST_PATH = Path("/content/eph_1t2026.manifest.json")
CHUNK_SIZE = 10_000

# Para probar sólo con Gran Córdoba, usá: AGLOMERADOS = {13}
AGLOMERADOS: set[int] = set()

# None significa 'todas'. Para achicar la prueba, pasá una lista de columnas.
# Ejemplo: ["CH04", "CH06", "ESTADO", "PONDERA"]
COLUMNAS_POR_TABLA: dict[str, list[str] | None] = {
    "individual": None,
    "hogar": None,
}


## 2. Bajar los datos

Primero intentamos bajar el ZIP desde el INDEC. Si esa página no responde desde Colab, usamos la copia del repositorio. También calculamos la huella SHA-256 para asegurarnos de que sea el mismo archivo con el que venimos trabajando.

In [12]:
def sha256_archivo(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for bloque in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(bloque)
    return digest.hexdigest()


def descargar(urls: list[str], destino: Path) -> str:
    temporal = destino.with_suffix(destino.suffix + ".part")
    errores: list[str] = []
    for url in urls:
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "eph-agente-colab/1.0"})
            with urllib.request.urlopen(request, timeout=120) as response, temporal.open("wb") as out:
                while bloque := response.read(1024 * 1024):
                    out.write(bloque)
            if not zipfile.is_zipfile(temporal):
                raise ValueError("la respuesta no es un ZIP válido")
            os.replace(temporal, destino)
            return url
        except Exception as exc:
            temporal.unlink(missing_ok=True)
            errores.append(f"{url}: {exc}")
    raise RuntimeError("No se pudo descargar la fuente:\n" + "\n".join(errores))


url_usada = descargar(URLS_FUENTE, ZIP_PATH)
zip_sha256 = sha256_archivo(ZIP_PATH)
if zip_sha256 != EXPECTED_ZIP_SHA256:
    raise RuntimeError(
        "El ZIP descargado cambió respecto de la fuente versionada. "
        "Revisá la actualización metodológica antes de continuar. "
        f"Esperado: {EXPECTED_ZIP_SHA256}; recibido: {zip_sha256}"
    )
print(f"Fuente: {url_usada}")
print(f"Archivo: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MiB)")
print(f"SHA-256: {zip_sha256}")


Fuente: https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/EPH_usu_1_Trim_2026_txt.zip
Archivo: /content/EPH_usu_1_Trim_2026_txt.zip (3.4 MiB)
SHA-256: d9d23945d4b1d3de362e154015fd9abacc76ed641aec6f41a805b809780c0343


## 3. Leer el archivo sin cargarlo entero

Leemos cada TXT en grupos de `CHUNK_SIZE` filas. En la primera pasada vemos qué tipo tiene cada columna y contamos los registros; en la segunda guardamos los datos en SQLite. Tarda un poco más que leer todo de una vez, pero nos permite usar el mismo código si más adelante el archivo crece.

In [13]:
TABLAS = ("individual", "hogar")
COLUMNAS_OBLIGATORIAS = {"ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "AGLOMERADO"}


def normalizar_nombre(nombre: str) -> str:
    return nombre.strip().upper().lstrip("\ufeff")


def citar(identificador: str) -> str:
    return '"' + identificador.replace('"', '""') + '"'


def localizar_miembros(zf: zipfile.ZipFile) -> dict[str, str]:
    encontrados: dict[str, str] = {}
    for nombre in zf.namelist():
        base = Path(nombre).name.lower()
        if not base.endswith(".txt"):
            continue
        for tabla in TABLAS:
            if tabla in base:
                if tabla in encontrados:
                    raise ValueError(f"Hay más de un TXT para {tabla}")
                encontrados[tabla] = nombre
    faltantes = set(TABLAS) - encontrados.keys()
    if faltantes:
        raise ValueError(f"Faltan tablas en el ZIP: {sorted(faltantes)}")
    return encontrados


def encabezado(zf: zipfile.ZipFile, miembro: str) -> list[str]:
    with zf.open(miembro) as raw, io.TextIOWrapper(raw, encoding="latin-1") as text:
        frame = pd.read_csv(text, sep=";", nrows=0)
    return [normalizar_nombre(col) for col in frame.columns]


def seleccionar_columnas(tabla: str, disponibles: list[str]) -> list[str]:
    solicitadas = COLUMNAS_POR_TABLA[tabla]
    requeridas = set(COLUMNAS_OBLIGATORIAS)
    if tabla == "individual":
        requeridas.add("COMPONENTE")
    seleccion = set(disponibles if solicitadas is None else map(normalizar_nombre, solicitadas)) | requeridas
    desconocidas = seleccion - set(disponibles)
    if desconocidas:
        raise ValueError(f"Columnas inexistentes en {tabla}: {sorted(desconocidas)}")
    return [col for col in disponibles if col in seleccion]


def iterar_chunks(zf: zipfile.ZipFile, miembro: str, columnas: list[str]):
    seleccion = set(columnas)
    with zf.open(miembro) as raw, io.TextIOWrapper(raw, encoding="latin-1") as text:
        lector = pd.read_csv(
            text,
            sep=";",
            dtype=str,
            keep_default_na=False,
            chunksize=CHUNK_SIZE,
            usecols=lambda col: normalizar_nombre(col) in seleccion,
        )
        for frame in lector:
            frame.columns = [normalizar_nombre(col) for col in frame.columns]
            yield frame[columnas]


def filtrar(frame: pd.DataFrame) -> pd.DataFrame:
    if not AGLOMERADOS:
        return frame
    codigos = {str(codigo) for codigo in AGLOMERADOS}
    return frame[frame["AGLOMERADO"].str.strip().isin(codigos)]


def validar_periodo(frame: pd.DataFrame, tabla: str) -> None:
    if frame.empty:
        return
    periodos = set(zip(frame["ANO4"].str.strip(), frame["TRIMESTRE"].str.strip()))
    if periodos != {("2026", "1")}:
        raise ValueError(f"{tabla} contiene períodos fuera de 2026T1: {sorted(periodos)}")


def perfilar_tabla(zf: zipfile.ZipFile, tabla: str, miembro: str, columnas: list[str]):
    estado = {col: {"numerica": True, "entera": True, "con_valor": False} for col in columnas}
    total = 0
    for frame in iterar_chunks(zf, miembro, columnas):
        frame = filtrar(frame)
        validar_periodo(frame, tabla)
        total += len(frame)
        for col in columnas:
            valores = frame[col].str.strip()
            valores = valores[valores.ne("")]
            if valores.empty:
                continue
            estado[col]["con_valor"] = True
            if col == "CODUSU" or col.endswith("_COD"):
                estado[col]["numerica"] = False
                continue
            numeros = pd.to_numeric(valores.str.replace(",", ".", regex=False), errors="coerce")
            if numeros.isna().any():
                estado[col]["numerica"] = False
                estado[col]["entera"] = False
            elif not (numeros % 1 == 0).all():
                estado[col]["entera"] = False
    if total == 0:
        raise ValueError(f"El filtro dejó {tabla} sin filas")
    tipos = {}
    for col, info in estado.items():
        if info["numerica"] and info["con_valor"]:
            tipos[col] = "INTEGER" if info["entera"] else "REAL"
        else:
            tipos[col] = "TEXT"
    return tipos, total


def convertir(frame: pd.DataFrame, tipos: dict[str, str]) -> pd.DataFrame:
    salida = frame.copy()
    for col, tipo in tipos.items():
        valores = salida[col].str.strip()
        vacios = valores.eq("")
        if tipo == "TEXT":
            salida[col] = valores.where(~vacios, None)
        else:
            numeros = pd.to_numeric(valores.str.replace(",", ".", regex=False), errors="raise")
            numeros[vacios] = pd.NA
            salida[col] = numeros.astype("Int64" if tipo == "INTEGER" else "Float64")
    return salida


def valor_sql(valor: object) -> object:
    if pd.isna(valor):
        return None
    return valor.item() if hasattr(valor, "item") else valor


def construir_sqlite(zip_path: Path, db_path: Path) -> dict:
    temporal = db_path.with_suffix(".tmp.sqlite")
    temporal.unlink(missing_ok=True)
    perfiles: dict[str, dict] = {}
    with zipfile.ZipFile(zip_path) as zf:
        miembros = localizar_miembros(zf)
        for tabla, miembro in miembros.items():
            disponibles = encabezado(zf, miembro)
            columnas = seleccionar_columnas(tabla, disponibles)
            tipos, filas = perfilar_tabla(zf, tabla, miembro, columnas)
            perfiles[tabla] = {"miembro": miembro, "columnas": columnas, "tipos": tipos, "filas": filas}

        con = sqlite3.connect(temporal)
        try:
            con.execute("PRAGMA journal_mode = MEMORY")
            con.execute("PRAGMA synchronous = OFF")
            with con:
                for tabla, perfil in perfiles.items():
                    definiciones = ", ".join(f"{citar(col)} {tipo}" for col, tipo in perfil["tipos"].items())
                    con.execute(f"CREATE TABLE {citar(tabla)} ({definiciones})")
                    columnas_sql = ", ".join(map(citar, perfil["columnas"]))
                    marcas = ", ".join("?" for _ in perfil["columnas"])
                    insert = f"INSERT INTO {citar(tabla)} ({columnas_sql}) VALUES ({marcas})"
                    insertadas = 0
                    for frame in iterar_chunks(zf, perfil["miembro"], perfil["columnas"]):
                        frame = convertir(filtrar(frame), perfil["tipos"])
                        filas = (tuple(valor_sql(v) for v in fila) for fila in frame.itertuples(index=False, name=None))
                        con.executemany(insert, filas)
                        insertadas += len(frame)
                    if insertadas != perfil["filas"]:
                        raise RuntimeError(f"Conteo inconsistente en {tabla}: {insertadas} != {perfil['filas']}")
                    con.execute(f"CREATE INDEX ix_{tabla}_hogar ON {citar(tabla)} (CODUSU, NRO_HOGAR)")
                    con.execute(f"CREATE INDEX ix_{tabla}_aglo ON {citar(tabla)} (AGLOMERADO)")
            con.execute("VACUUM")
        finally:
            con.close()
    os.replace(temporal, db_path)
    return perfiles


## 4. Armar la base

Además de la base, guardamos un archivo JSON con la URL usada, la huella del ZIP, los filtros y la cantidad de filas. Así después podemos saber exactamente cómo se generó.

In [35]:
perfiles = construir_sqlite(ZIP_PATH, DB_PATH)

modo_completo = not AGLOMERADOS and all(valor is None for valor in COLUMNAS_POR_TABLA.values())
manifest = {
    "dataset": "EPH continua 1T 2026",
    "fuente": url_usada,
    "zip_sha256": zip_sha256,
    "generado_utc": datetime.now(timezone.utc).isoformat(),
    "modo": "completo_canonico" if modo_completo else "filtrado_exploratorio",
    "filtros": {"aglomerados": sorted(AGLOMERADOS)},
    "chunk_size": CHUNK_SIZE,
    "tablas": {
        tabla: {"filas": perfil["filas"], "columnas": perfil["columnas"], "tipos": perfil["tipos"]}
        for tabla, perfil in perfiles.items()
    },
    "advertencias": [
        "Una base filtrada cambia el universo; toda estimación debe documentar filtro, universo y ponderador."
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({tabla: datos["filas"] for tabla, datos in manifest["tablas"].items()}, indent=2))
print(f"SQLite: {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MiB)")
print(f"Manifiesto: {MANIFEST_PATH}")


{
  "hogar": 15447,
  "individual": 43739
}
SQLite: /content/eph_1t2026.sqlite (21.1 MiB)
Manifiesto: /content/eph_1t2026.manifest.json


## 5. Un chequeo rápido

Comprobamos que estén las dos tablas y miramos cinco filas. Si no usamos filtros, también verificamos los conteos que ya conocemos para el 1T 2026.

In [15]:
def ejecutar_consulta(sql: str, params=None, display_result=False) -> pd.DataFrame:
    with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as con:
        result = pd.read_sql_query(sql, con)
        if display_result:
            display(result)
    return result

conteos = {tabla: ejecutar_consulta(f"SELECT COUNT(*) FROM {citar(tabla)}").iloc[0, 0] for tabla in TABLAS}

muestra = ejecutar_consulta("SELECT ANO4, TRIMESTRE, AGLOMERADO, CODUSU, NRO_HOGAR, COMPONENTE FROM individual LIMIT 5")

print(conteos)
display(muestra)


{'individual': np.int64(43739), 'hogar': np.int64(15447)}


,ANO4,TRIMESTRE,AGLOMERADO,CODUSU,NRO_HOGAR,COMPONENTE
0,2026,1,93,TQRMNORURHLLKUCDEOJAH00893620,1,2
1,2026,1,93,TQRMNORURHLLKUCDEOJAH00893620,1,1
2,2026,1,93,TQRMNORURHLLKUCDEOJAH00893620,1,3
3,2026,1,93,TQRMNORURHLLKUCDEOJAH00893620,1,4
4,2026,1,25,TQRMNOTUQHJOLOCDEHLEH00875560,1,3


## 6. Guardar el resultado

Por ahora SQLite nos alcanza: es simple de bajar, abrir y volver a generar. Si después sumamos varios períodos o necesitamos que varias personas consulten al mismo tiempo, podremos llevar esta misma carga a PostgreSQL. Antes de agregar otro trimestre habrá que comparar el diseño de registros del INDEC.

In [16]:
try:
    from google.colab import files
    files.download(str(DB_PATH))
    files.download(str(MANIFEST_PATH))
except ImportError:
    print("Fuera de Colab; archivos disponibles en:", DB_PATH, MANIFEST_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Los datos de la Encuesta Permanente de Hogares (EPH) no surgen de una muestra de hogares, no de contar a toda la población. A cada persona encuestada se le asigna un peso, llamado ponderador, que indica a cuántas personas representa, y las cifras se obtienen sumando esos pesos. Además, la encuesta se realiza solo en los 31 principales aglomerados urbanos del país, por lo que sus resultados describen a esas ciudades y no al total de la Argentina: quedan afuera las localidades más chicas y las zonas rurales. Así, cuando se dice que la población alcanzada es de 30,1 millones de personas y que 14,6 millones forman parte de la población económicamente activa, esas cifras se refieren a los 31 aglomerados relevados, no a todo el país.

Glosario:
- **Ocupado**: Tienen al menos una ocupación (trabajaron al menos una hora)
- **Desocupado**: No tienen ocupación pero buscan trabajo activamente y están disponibles para trabajar
- **Población Económicamente Activa (PEA)**: Tienen al menos una ocupación o, sin tenerla, la buscan activamente y están disponibles para trabajar

Preguntas:

1. Cuál es la población alcanzada? 30.1 millones de personas
2. Qué proporción de la población es ocupada? 44.8%
3. Qué proporción de los ocupados son formales? 55.7%
4. Qué proporción de los ocupados son varones? 55.3%
5. Qué proporción de los ocupados son asalariados? 71.8%
6. Qué proporción de los asalariados es formal? 62.1%
7. Qué proporción de los no asalariados es cuenta propista? 85.8%
8. Qué proporción de los ocupados busca trabajo? 17.1%
9. Qué proporción de la PEA es desocupada? 7.8%
10. Qué proporción de los desocupados tienen secundario incompleto? 35.9%
11. Cúal es aglomerado con mayor población ? Partidos del Gran Buenos Aires 13.229.175
12. Cúal es el aglomerado con mayor tasa de desocupación? San Nicolás-Villa Constitución, 10.3%

Fuente: [Mercado de trabajo. Tasas e indicadores socioeconómicos (EPH)](https://www.indec.gob.ar/uploads/informesdeprensa/ingresos1trim26536C8EDD3E.pdf).

In [17]:
# Cuál es la población alcanzada?

_ = ejecutar_consulta("""
SELECT SUM(PONDERA) AS poblacion_alcanzada
FROM individual
""", display_result=True)


,poblacion_alcanzada
0,30095103


,tasa_empleo
0,0.45


In [18]:
# Qué proporción de la población es ocupada?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN ESTADO = 1 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS tasa_empleo
FROM individual;
""", display_result=True)

,tasa_empleo
0,0.45


In [20]:
# Qué proporción de los ocupados son formales?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN EMPLEO = 1 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_formales
FROM individual
WHERE ANO4 = 2026 AND TRIMESTRE = 1
  AND ESTADO = 1;          -- solo ocupados
""", display_result=True)


,proporcion_formales
0,0.56


In [22]:
# Qué proporción de los ocupados son varones?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN CH04 = 1 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_varones
FROM individual
WHERE ESTADO = 1;          -- solo ocupados
""", display_result=True)

,proporcion_varones
0,0.55


In [23]:
# Qué proporción de los ocupados son asalariados?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN CAT_OCUP = 3 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_asalariados
FROM individual
WHERE ANO4 = 2026 AND TRIMESTRE = 1
  AND ESTADO = 1;          -- solo ocupados
""", display_result=True)

In [25]:
# Qué proporción de los asalariados es formal?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN EMPLEO = 1 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_asalariados_formales
FROM individual
WHERE
  ESTADO = 1           -- ocupados
  AND CAT_OCUP = 3;        -- asalariados
""", display_result=True)

,proporcion_asalariados_formales
0,0.62


In [29]:
# Qué proporción de los no asalariados es cuenta propista?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN CAT_OCUP = 2 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_cuentapropistas
FROM individual
WHERE
ESTADO = 1           -- ocupados
  AND CAT_OCUP <> 3;       -- no asalariados
""", display_result=True)

,proporcion_cuentapropistas
0,0.86


In [37]:
#Qué proporción de los ocupados busca trabajo?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN PP03J = 1 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_ocupados_que_buscan
FROM individual
WHERE ESTADO = 1;          -- solo ocupados""", display_result=True)

,proporcion_ocupados_que_buscan
0,0.17


In [40]:
# Qué proporción de la PEA es desocupada?
_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN ESTADO = 2 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS tasa_desocupacion
FROM individual
WHERE  ESTADO IN (1, 2);    -- PEA: ocupados y desocupados
""", display_result=True)

,tasa_desocupacion
0,0.08


In [42]:
# cual es el aglomerado con mayor tasa de desocupación?

_ = ejecutar_consulta("""
SELECT
  AGLOMERADO,
  SUM(CASE WHEN ESTADO = 2 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS tasa_desocupacion
FROM individual
WHERE  ESTADO IN (1, 2)     -- PEA
GROUP BY AGLOMERADO
ORDER BY tasa_desocupacion DESC
LIMIT 1;
""", display_result=True)

,AGLOMERADO,tasa_desocupacion
0,38,0.10


In [44]:
# Qué proporción de los desocupados tienen secundario incompleto?

_ = ejecutar_consulta("""
SELECT
  SUM(CASE WHEN NIVEL_ED = 3 THEN PONDERA END) * 1.0
    / SUM(PONDERA) AS proporcion_secundario_incompleto
FROM individual
WHERE
  ESTADO = 2;          -- solo desocupados

""", display_result=True)

,proporcion_secundario_incompleto
0,0.22


**EDA**

# Columnas candidatas:

- ESTADO
- PP0
-